In [ ]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor
import lightgbm as lgb

# ==============================
# 2. LOAD DATA
# ==============================
df = pd.read_csv("house_prices.csv")

# ==============================
# 3. TARGET + LOG TRANSFORM
# ==============================
y = np.log1p(df['SalePrice'])   # 🔥 important
X = df.drop('SalePrice', axis=1)

# ==============================
# 4. MISSING VALUES
# ==============================
# Numeric
num_cols = X.select_dtypes(include=['int64','float64']).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

# Categorical
cat_cols = X.select_dtypes(include=['object']).columns
X[cat_cols] = X[cat_cols].fillna("Missing")

# ==============================
# 5. FEATURE ENGINEERING
# ==============================
# Total area feature
if 'GrLivArea' in X.columns and 'TotalBsmtSF' in X.columns:
    X['TotalArea'] = X['GrLivArea'] + X['TotalBsmtSF']

# Age of house
if 'YearBuilt' in X.columns:
    X['HouseAge'] = 2024 - X['YearBuilt']

# ==============================
# 6. ENCODING
# ==============================
X = pd.get_dummies(X, drop_first=True)

# ==============================
# 7. SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 8. RANDOM FOREST BASELINE
# ==============================
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print("RF RMSE:", rf_rmse)

# ==============================
# 9. XGBOOST (EARLY STOPPING)
# ==============================
xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=50,
    verbose=False
)

xgb_pred = xgb.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

print("XGB RMSE:", xgb_rmse)

# ==============================
# 10. LIGHTGBM
# ==============================
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgb_model.fit(X_train, y_train)

lgb_pred = lgb_model.predict(X_test)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))

print("LGBM RMSE:", lgb_rmse)

# ==============================
# 11. STACKING MODEL
# ==============================
stack = StackingRegressor(
    estimators=[
        ('rf', rf),
        ('xgb', xgb),
        ('lgb', lgb_model)
    ],
    final_estimator=Ridge()
)

stack.fit(X_train, y_train)

stack_pred = stack.predict(X_test)
stack_rmse = np.sqrt(mean_squared_error(y_test, stack_pred))

print("STACK RMSE:", stack_rmse)

# ==============================
# 12. FINAL COMPARISON
# ==============================
print("\nMODEL COMPARISON:")
print(f"RF:   {rf_rmse}")
print(f"XGB:  {xgb_rmse}")
print(f"LGB:  {lgb_rmse}")
print(f"STACK:{stack_rmse}")